In [ ]:
import numpy as np
import cupy as cp
from scipy import ndimage
from types import SimpleNamespace
from scipy.fft import fftn, ifftn  # faster than np.fft on CPU

from holotomocupy.rec import Rec
from holotomocupy.shift import Shift
from holotomocupy.tomo_batched import TomoBatched
from holotomocupy.utils import *

# Acquisition parameters

In [ ]:
n = 128
ntheta = 360
detector_pixelsize = 1.4760147601476e-6 * 2 * 2048/n
energy = 17.1
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.217

sx0 = -3.135e-3
z1 = np.array([5.110, 5.464, 6.879, 9.817, 10.372, 11.146, 12.594, 17.209]) * 1e-3 - sx0
z1_ids = np.array([0, 1, 2, 3]) ### note using index starting from 0
z1 = z1[z1_ids]
ndist = len(z1)
z2 = focustodetectordistance - z1
distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]

nobj = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 


# Modeling data

In [ ]:
cl = Shift(n,nobj,n,nobj,1/norm_magnifications)

a = (cp.random.random([1,nobj,nobj])+1j*cp.random.random([1,nobj,nobj])).astype('complex64')
b = (cp.random.random([1,n,n])+1j*cp.random.random([1,n,n])).astype('complex64')
pos = (cp.random.random([1,4,2])).astype('float32')
bb=cl.curlyS(a,pos,3)
aa=cl.curlySadj(b,pos,3)
print(cp.sum(a*cp.conj(aa)))
print(cp.sum(b*cp.conj(bb)))


In [ ]:
pad=n//2
a = cp.zeros([1,nobj,nobj]).astype('complex64')
a[:,pad:-pad,pad:-pad] = (1+1j/10)*1e-2
b = cl.curlyS(a,pos,3)
c = cp.exp(1j*b)

bb = cp.exp(1j*a)
cc = cl.curlyS(bb,pos,3)

mshow_complex(a[0],True)

mshow_complex(c[0],True)
mshow_complex(c[0,n//4:-n//4,n//4:-n//4],True)

# mshow_complex(a[0],True)
# mshow_complex(bb[0],True)
mshow_complex(cc[0],True)

mshow_complex(c[0]-cc[0],True)

# plt.plot(c[0,n//2,n//4:-n//4].imag.get())
# plt.plot(c[0,n//2,n//4:-n//4].imag.get()-cc[0,n//2,n//4:-n//4].imag.get())
# a = cp.ones([1,n,n]).astype('complex64')
# b=cl.curlySadj(a,pos,3)
# mshow(b[0].real,True)
# mshow(b[0,nobj//4:-nobj//4,nobj//4:-nobj//4].real,True)

